In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

samples = [
    # add your samples here
    "DRG_0198_01",
    "DRG_0199_01",
    "DRG_0341_01",
    "DRG_1409_01",
    "DRG_1215_02",
    "DRG_2324_02",
    "DRG_2902_02",
    "DRG_5467_02",
]

base = Path("/home/913/dk4874/scratch/gdata/scDaisychain_paper/human/processed_data/whatshap")

plot_dir = base / "phase_concordance_plots"
plot_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Load all phase-set concordance files
# ------------------------------------------------------------

dfs = []

for sample in samples:
    f = base / sample / "comparison_qual20gq10" / "phase_set_concordance.tsv"

    if not f.exists():
        print(f"[WARN] Missing: {f}")
        continue

    df = pd.read_csv(f, sep="\t")
    df["sample"] = sample
    dfs.append(df)

if not dfs:
    raise FileNotFoundError("No phase_set_concordance.tsv files found.")

all_df = pd.concat(dfs, ignore_index=True)

# Ensure numeric columns are numeric
num_cols = [
    "start", "end", "span_bp",
    "n_shared_snps",
    "n_same_orientation",
    "n_flipped_orientation",
    "n_concordant_sites_after_best_flip",
    "n_discordant_sites_after_best_flip",
    "site_concordance_after_best_flip",
    "total_snp_pairs",
    "concordant_snp_pairs",
    "pairwise_phase_concordance",
    "pairwise_phase_discordance",
    "n_orientation_switches",
    "switch_rate",
]

for c in num_cols:
    if c in all_df.columns:
        all_df[c] = pd.to_numeric(all_df[c], errors="coerce")

# Clean useful subset
plot_df = all_df.dropna(subset=["sample", "pairwise_phase_concordance", "total_snp_pairs"]).copy()

# ------------------------------------------------------------
# Per-sample summary table
# ------------------------------------------------------------

summary_rows = []

for sample, sub in plot_df.groupby("sample"):
    sub_pairs = sub[sub["total_snp_pairs"] > 0].copy()
    sub_sites = sub[sub["n_shared_snps"] > 0].copy()

    if len(sub_pairs) > 0:
        weighted_pairwise_concordance = np.average(
            sub_pairs["pairwise_phase_concordance"],
            weights=sub_pairs["total_snp_pairs"],
        )
        total_snp_pairs = sub_pairs["total_snp_pairs"].sum()
        concordant_snp_pairs = sub_pairs["concordant_snp_pairs"].sum()
    else:
        weighted_pairwise_concordance = np.nan
        total_snp_pairs = 0
        concordant_snp_pairs = 0

    if len(sub_sites) > 0:
        weighted_site_concordance = np.average(
            sub_sites["site_concordance_after_best_flip"],
            weights=sub_sites["n_shared_snps"],
        )
        total_shared_snps = sub_sites["n_shared_snps"].sum()
    else:
        weighted_site_concordance = np.nan
        total_shared_snps = 0

    total_span_bp = sub["span_bp"].sum(skipna=True)
    n_phase_sets = len(sub)

    summary_rows.append({
        "sample": sample,
        "n_phase_sets": n_phase_sets,
        "total_span_bp": total_span_bp,
        "total_shared_snps": total_shared_snps,
        "total_snp_pairs": total_snp_pairs,
        "concordant_snp_pairs": concordant_snp_pairs,
        "weighted_site_concordance": weighted_site_concordance,
        "weighted_pairwise_phase_concordance": weighted_pairwise_concordance,
        "mean_pairwise_phase_concordance": sub["pairwise_phase_concordance"].mean(),
        "median_pairwise_phase_concordance": sub["pairwise_phase_concordance"].median(),
        "total_orientation_switches": sub["n_orientation_switches"].sum(skipna=True),
    })

summary = pd.DataFrame(summary_rows)

summary["weighted_pairwise_phase_discordance"] = (
    1 - summary["weighted_pairwise_phase_concordance"]
)

summary = summary.sort_values("weighted_pairwise_phase_concordance", ascending=False)

summary_file = plot_dir / "phase_concordance_summary_by_sample.tsv"
summary.to_csv(summary_file, sep="\t", index=False)

print(summary)
print(f"\nSaved summary: {summary_file}")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

samples = [
    # add your samples here
    "DRG_0198_01",
    "DRG_0199_01",
    "DRG_0341_01",
    "DRG_1409_01",
    "DRG_1215_02",
    "DRG_2324_02",
    "DRG_2902_02",
    "DRG_5467_02",
]

base = Path("/home/913/dk4874/scratch/gdata/scDaisychain_paper/human/processed_data/whatshap")

plot_dir = base / "phase_concordance_plots"
plot_dir.mkdir(parents=True, exist_ok=True)

# Sliding window settings
window_size_bp = 2_500_000   # 2.5 Mb
step_bp = 1_250_000          # 1.25 Mb
min_phase_sets_per_sample_window = 1
min_samples_per_window = 1

# Minimum shared SNPs per phase set
min_shared_snps = 2

# ------------------------------------------------------------
# Coordinates: GRCh38 / hg38
# ------------------------------------------------------------

CHR_X_LEN = 156_040_895

# PAR1: chrX:10,001-2,781,479
PAR1_START = 10_001
PAR1_END = 2_781_479

# XIST: chrX:73,820,649-73,852,714
XIST_START = 73_820_649
XIST_END = 73_852_714

xscale = 1e6  # bp to Mb

def fmt_mb(bp):
    """Format bp as Mb without forced 1-decimal rounding."""
    return f"{bp / 1e6:g}"

# ------------------------------------------------------------
# Load all phase-set concordance files
# ------------------------------------------------------------

dfs = []

for sample in samples:
    f = base / sample / "comparison_qual20gq10" / "phase_set_concordance.tsv"

    if not f.exists():
        print(f"[WARN] Missing: {f}")
        continue

    df = pd.read_csv(f, sep="\t")
    df["sample"] = sample
    dfs.append(df)

if not dfs:
    raise FileNotFoundError("No phase_set_concordance.tsv files found.")

all_df = pd.concat(dfs, ignore_index=True)

# ------------------------------------------------------------
# Ensure numeric columns are numeric
# ------------------------------------------------------------

num_cols = [
    "start", "end", "span_bp",
    "n_shared_snps",
    "n_same_orientation",
    "n_flipped_orientation",
    "n_concordant_sites_after_best_flip",
    "n_discordant_sites_after_best_flip",
    "site_concordance_after_best_flip",
    "total_snp_pairs",
    "concordant_snp_pairs",
    "pairwise_phase_concordance",
    "pairwise_phase_discordance",
    "n_orientation_switches",
    "switch_rate",
]

for c in num_cols:
    if c in all_df.columns:
        all_df[c] = pd.to_numeric(all_df[c], errors="coerce")

# ------------------------------------------------------------
# Clean dataframe
# ------------------------------------------------------------

required_cols = [
    "sample",
    "start",
    "end",
    "n_shared_snps",
    "site_concordance_after_best_flip",
]

plot_df = all_df.dropna(subset=required_cols).copy()

# Exclude phase sets with fewer than 2 shared SNPs
plot_df = plot_df[plot_df["n_shared_snps"] >= min_shared_snps].copy()

# Clip intervals to chrX bounds
plot_df["start"] = plot_df["start"].clip(lower=1, upper=CHR_X_LEN)
plot_df["end"] = plot_df["end"].clip(lower=1, upper=CHR_X_LEN)

# Ensure start <= end
bad = plot_df["end"] < plot_df["start"]
plot_df.loc[bad, ["start", "end"]] = plot_df.loc[bad, ["end", "start"]].values

# Keep only samples with data
sample_order = [s for s in samples if s in plot_df["sample"].unique()]

if not sample_order:
    raise ValueError(
        f"No samples with usable phase-set concordance data after filtering to "
        f"n_shared_snps >= {min_shared_snps}."
    )

# ------------------------------------------------------------
# Build sample-level sliding windows
# ------------------------------------------------------------

window_starts = np.arange(
    1,
    CHR_X_LEN - window_size_bp + 2,
    step_bp,
)

# Add final window so end of chrX is covered
final_start = CHR_X_LEN - window_size_bp + 1

if window_starts[-1] != final_start:
    window_starts = np.append(window_starts, final_start)

window_rows = []

for sample in sample_order:
    sub = plot_df[plot_df["sample"] == sample].copy()

    for wstart in window_starts:
        wend = min(wstart + window_size_bp - 1, CHR_X_LEN)

        # Phase set contributes if it overlaps the window
        overlap = sub[
            (sub["end"] >= wstart) &
            (sub["start"] <= wend)
        ]

        n_phase_sets = len(overlap)

        if n_phase_sets >= min_phase_sets_per_sample_window:
            mean_conc = overlap["site_concordance_after_best_flip"].mean()
            median_conc = overlap["site_concordance_after_best_flip"].median()
        else:
            mean_conc = np.nan
            median_conc = np.nan

        window_rows.append({
            "sample": sample,
            "window_start": wstart,
            "window_end": wend,
            "window_mid": (wstart + wend) / 2,
            "n_phase_sets": n_phase_sets,
            "mean_site_concordance_after_best_flip": mean_conc,
            "median_site_concordance_after_best_flip": median_conc,
        })

window_df = pd.DataFrame(window_rows)

sample_window_file = (
    plot_dir /
    f"phase_set_site_concordance_after_best_flip_sliding_windows_by_sample_min{min_shared_snps}snps.tsv"
)
window_df.to_csv(sample_window_file, sep="\t", index=False)

print(f"Saved sample-level sliding-window table: {sample_window_file}")

# ------------------------------------------------------------
# Average across samples per window
# ------------------------------------------------------------

window_avg = (
    window_df
    .dropna(subset=["mean_site_concordance_after_best_flip"])
    .groupby(["window_start", "window_end", "window_mid"], as_index=False)
    .agg(
        mean_across_samples=(
            "mean_site_concordance_after_best_flip",
            "mean",
        ),
        median_across_samples=(
            "mean_site_concordance_after_best_flip",
            "median",
        ),
        sd_across_samples=(
            "mean_site_concordance_after_best_flip",
            "std",
        ),
        n_samples_with_data=(
            "sample",
            "nunique",
        ),
        total_phase_sets_in_window=(
            "n_phase_sets",
            "sum",
        ),
    )
)

window_avg = window_avg[
    window_avg["n_samples_with_data"] >= min_samples_per_window
].copy()

if window_avg.empty:
    raise ValueError("No windows passed min_samples_per_window filtering.")

# SD is NaN when only one sample contributes to a window.
# Fill with 0 so the band collapses to the mean rather than breaking the plot.
window_avg["sd_across_samples"] = window_avg["sd_across_samples"].fillna(0)

# Mean ± 1 SD, clipped to valid concordance range
window_avg["sd_lower"] = (
    window_avg["mean_across_samples"] - window_avg["sd_across_samples"]
).clip(lower=0, upper=1)

window_avg["sd_upper"] = (
    window_avg["mean_across_samples"] + window_avg["sd_across_samples"]
).clip(lower=0, upper=1)

avg_window_file = (
    plot_dir /
    f"phase_set_site_concordance_after_best_flip_sliding_windows_average_across_samples_min{min_shared_snps}snps.tsv"
)
window_avg.to_csv(avg_window_file, sep="\t", index=False)

print(f"Saved average sliding-window table: {avg_window_file}")

# ------------------------------------------------------------
# Plot average across samples with SD shading
# ------------------------------------------------------------

plot_avg = window_avg.sort_values("window_mid").copy()

x = plot_avg["window_mid"].to_numpy() / xscale
y = plot_avg["mean_across_samples"].to_numpy()
y_lower = plot_avg["sd_lower"].to_numpy()
y_upper = plot_avg["sd_upper"].to_numpy()

fig, ax = plt.subplots(figsize=(16, 5.5))

# Shaded genomic regions
ax.axvspan(
    PAR1_START / xscale,
    PAR1_END / xscale,
    color="khaki",
    alpha=0.45,
    zorder=0,
)

ax.axvspan(
    XIST_START / xscale,
    XIST_END / xscale,
    color="mistyrose",
    alpha=0.9,
    zorder=0,
)

# SD shading + mean horizontal line for each sliding window
for _, row in plot_avg.iterrows():

    x0 = row["window_start"] / xscale
    x1 = row["window_end"] / xscale

    y_mean = row["mean_across_samples"]
    y_lower = row["sd_lower"]
    y_upper = row["sd_upper"]

    # SD shading across the full window span
    ax.fill_between(
        [x0, x1],
        [y_lower, y_lower],
        [y_upper, y_upper],
        color="#4C78A8",
        alpha=0.22,
        linewidth=0,
        zorder=1,
    )

    # Mean horizontal line across the full window span
    ax.plot(
        [x0, x1],
        [y_mean, y_mean],
        color="#4C78A8",
        linewidth=2.2,
        alpha=0.9,
        solid_capstyle="butt",
        zorder=2,
    )

# Optional: show the original sliding-window horizontal segments faintly
# This makes it clear each value represents a full window span.
for _, row in plot_avg.iterrows():
    ax.plot(
        [row["window_start"] / xscale, row["window_end"] / xscale],
        [row["mean_across_samples"], row["mean_across_samples"]],
        color="#4C78A8",
        linewidth=1.0,
        alpha=0.25,
        solid_capstyle="butt",
        zorder=2,
    )

# Region labels
ax.text(
    ((PAR1_START + PAR1_END) / 2) / xscale,
    1.015,
    "PAR1",
    ha="center",
    va="bottom",
    fontsize=9,
    color="dimgray",
)

ax.text(
    ((XIST_START + XIST_END) / 2) / xscale,
    1.015,
    "XIST",
    ha="center",
    va="bottom",
    fontsize=9,
    color="dimgray",
)

# Formatting
ax.set_xlim(0, CHR_X_LEN / xscale)
ax.set_ylim(-0.02, 1.02)

ax.set_xlabel("chrX position (Mb)")
ax.set_ylabel("Mean site concordance after best flip")

ax.set_title(
    "Sliding-window phase-set site concordance across human chrX"
)

ax.set_xticks(np.arange(0, CHR_X_LEN / xscale + 1, 20))
ax.grid(axis="y", alpha=0.25)

# Legend
handles = [
    Line2D(
        [0],
        [0],
        color="#4C78A8",
        linewidth=2.2,
        label="Mean",
    ),
    Patch(
        facecolor="#4C78A8",
        edgecolor="none",
        alpha=0.22,
        label="±1 SD",
    ),
    Patch(
        facecolor="khaki",
        edgecolor="none",
        alpha=0.45,
        label="PAR1",
    ),
    Patch(
        facecolor="mistyrose",
        edgecolor="none",
        alpha=0.9,
        label="XIST",
    ),
]

ax.legend(
    handles=handles,
    loc="upper left",
    bbox_to_anchor=(1.01, 1.00),
    frameon=True,
)

plt.tight_layout(rect=[0, 0, 0.84, 1])

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

fig_file_png = (
    plot_dir /
    f"phase_set_site_concordance_after_best_flip_sliding_window_average_across_samples_PAR1_XIST_SDshade_min{min_shared_snps}snps.png"
)
fig_file_pdf = (
    plot_dir /
    f"phase_set_site_concordance_after_best_flip_sliding_window_average_across_samples_PAR1_XIST_SDshade_min{min_shared_snps}snps.pdf"
)
fig_file_svg = (
    plot_dir /
    f"phase_set_site_concordance_after_best_flip_sliding_window_average_across_samples_PAR1_XIST_SDshade_min{min_shared_snps}snps.svg"
)

fig.savefig(fig_file_png, dpi=300, bbox_inches="tight")
fig.savefig(fig_file_pdf, bbox_inches="tight")
fig.savefig(fig_file_svg, bbox_inches="tight")

plt.show()

print(f"Saved plot: {fig_file_png}")
print(f"Saved plot: {fig_file_pdf}")
print(f"Saved plot: {fig_file_svg}")

In [ ]:
# ------------------------------------------------------------
# Reverse ECDF / cumulative concordance plots
# includes PAR
# excludes phase sets with <2 shared SNPs
#
# Outputs:
#   1) one line per donor
#   2) donor mean + shaded SD
#
# Threshold axis runs from 50 to 100
# Highlights x = 90 and x = 100
#
# Compact 60 mm × 40 mm version
# ------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------
# Illustrator-editable / publication settings
# ------------------------------------------------------------

mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "font.family": "Arial",

    "font.size": 5.5,
    "axes.labelsize": 6.0,
    "xtick.labelsize": 5.5,
    "ytick.labelsize": 5.5,
    "legend.fontsize": 4.2,

    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.4,
    "ytick.major.size": 2.4,

    "savefig.transparent": True,
})

MM_TO_INCH = 1 / 25.4
FIG_W_MM = 60
FIG_H_MM = 40

FIG_W = FIG_W_MM * MM_TO_INCH
FIG_H = FIG_H_MM * MM_TO_INCH

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

plot_dir = Path(".")  # change if needed

metric_col = "site_concordance_after_best_flip"
min_shared_snps = 2

highlight_thresholds = [90, 100]

sample_labels = {
    "DRG_0198_01": "DRG_0198_01",
    "DRG_0199_01": "DRG_0199_01",
    "DRG_0341_01": "DRG_0341_01",
    "DRG_1409_01": "DRG_1409_01",
    "DRG_1215_02": "DRG_1215_02",
    "DRG_2324_02": "DRG_2324_02",
    "DRG_2902_02": "DRG_2902_02",
    "DRG_5467_02": "DRG_5467_02",
}

show_point_labels_by_sample = False
show_point_labels_mean = True

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

ecdf_df = plot_df.dropna(
    subset=["sample", "n_shared_snps", metric_col]
).copy()

ecdf_df = ecdf_df[ecdf_df["n_shared_snps"] >= min_shared_snps].copy()

if ecdf_df.empty:
    raise ValueError("No phase sets left after filtering.")

# Convert concordance to percent if stored as 0-1
if ecdf_df[metric_col].max() <= 1.5:
    ecdf_df[metric_col] = ecdf_df[metric_col] * 100

sample_order = [s for s in samples if s in ecdf_df["sample"].unique()]

if not sample_order:
    raise ValueError("No samples left after filtering phase sets.")

thresholds = np.linspace(50, 100, 501)

# ------------------------------------------------------------
# Summary per sample
# ------------------------------------------------------------

ecdf_summary = (
    ecdf_df
    .groupby("sample", as_index=False)
    .agg(
        n_phase_sets=(metric_col, "size"),
        mean_concordance_percent=(metric_col, "mean"),
        median_concordance_percent=(metric_col, "median"),
        sd_concordance_percent=(metric_col, "std"),
    )
)

ecdf_summary["sample_label"] = (
    ecdf_summary["sample"]
    .map(sample_labels)
    .fillna(ecdf_summary["sample"])
)

ecdf_summary_file = (
    plot_dir /
    f"phase_set_site_concordance_after_best_flip_reverse_ecdf_summary_min{min_shared_snps}snps_threshold50_PARincluded.tsv"
)
ecdf_summary.to_csv(ecdf_summary_file, sep="\t", index=False)

print(f"Saved summary: {ecdf_summary_file}")

# ------------------------------------------------------------
# Build reverse ECDF matrix
# ------------------------------------------------------------

curve_dict = {}
point_summary_rows = []

for sample in sample_order:
    vals = ecdf_df.loc[
        ecdf_df["sample"] == sample,
        metric_col
    ].dropna().values

    if len(vals) == 0:
        continue

    frac_ge = np.array([
        np.mean(vals >= t)
        for t in thresholds
    ]) * 100

    curve_dict[sample] = frac_ge

    row = {
        "sample": sample,
        "sample_label": sample_labels.get(sample, sample),
        "n_phase_sets": len(vals),
        "mean_concordance_percent": np.mean(vals),
    }

    for t in highlight_thresholds:
        row[f"pct_phase_sets_ge_{t}"] = np.mean(vals >= t) * 100

    point_summary_rows.append(row)

curve_df = pd.DataFrame(curve_dict, index=thresholds)
curve_df.index.name = "threshold_percent"

curve_file = (
    plot_dir /
    f"phase_set_site_concordance_after_best_flip_reverse_ecdf_curves_min{min_shared_snps}snps_threshold50_PARincluded.tsv"
)
curve_df.to_csv(curve_file, sep="\t")

print(f"Saved curve table: {curve_file}")

point_summary_df = pd.DataFrame(point_summary_rows)
point_summary_file = (
    plot_dir /
    f"phase_set_site_concordance_after_best_flip_reverse_ecdf_highlight_points_min{min_shared_snps}snps_threshold50_PARincluded.tsv"
)
point_summary_df.to_csv(point_summary_file, sep="\t", index=False)

print(f"Saved highlight-point summary: {point_summary_file}")

# ------------------------------------------------------------
# Plot 1: all donors as separate lines
# ------------------------------------------------------------

cmap = plt.get_cmap("tab10")
sample_colors = {
    sample: cmap(i % 10)
    for i, sample in enumerate(sample_order)
}

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))

fig.subplots_adjust(
    left=0.20,
    right=0.98,
    bottom=0.30,
    top=0.96,
)

for t in highlight_thresholds:
    ax.axvline(t, color="0.75", linestyle="--", linewidth=0.7, zorder=0)

for sample in sample_order:
    if sample not in curve_df.columns:
        continue

    vals = ecdf_df.loc[
        ecdf_df["sample"] == sample,
        metric_col
    ].dropna().values

    mean_val = np.mean(vals)
    y90 = np.mean(vals >= 90) * 100
    y100 = np.mean(vals >= 100) * 100

    ax.plot(
        thresholds,
        curve_df[sample].values,
        label=(
            f"{sample_labels.get(sample, sample)}  "
            f"n={len(vals):,}, mean={mean_val:.1f}%, "
            f"90={y90:.1f}%, 100={y100:.1f}%"
        ),
        color=sample_colors[sample],
        linewidth=1.15,
        alpha=0.95,
    )

    ax.scatter(
        [90, 100],
        [y90, y100],
        color=sample_colors[sample],
        s=14,
        zorder=4,
        edgecolor="white",
        linewidth=0.35,
    )

    if show_point_labels_by_sample:
        ax.annotate(
            f"{y90:.1f}%",
            xy=(90, y90),
            xytext=(2, 2),
            textcoords="offset points",
            fontsize=4.6,
            color=sample_colors[sample],
        )
        ax.annotate(
            f"{y100:.1f}%",
            xy=(100, y100),
            xytext=(-28, 2),
            textcoords="offset points",
            fontsize=4.6,
            color=sample_colors[sample],
            ha="right",
        )

# Slight extension so the 100% points are not clipped
ax.set_xlim(50, 101.5)
ax.set_ylim(0, 105)

ax.set_xlabel("Concordance threshold (%)", labelpad=2)
ax.set_ylabel("Phase sets ≥ threshold (%)", labelpad=2)

ax.set_xticks([50, 60, 70, 80, 90, 100])
ax.set_yticks([0, 25, 50, 75, 100])

ax.grid(axis="y", alpha=0.22, linewidth=0.4)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    loc="lower left",
    bbox_to_anchor=(0.03, 0.03),
    ncol=1,
    frameon=True,
    facecolor="white",
    edgecolor="0.8",
    framealpha=0.88,
    handlelength=1.2,
    handletextpad=0.35,
    borderpad=0.35,
    borderaxespad=0.0,
    labelspacing=0.30,
    prop={"size": 3.8},
)

fig_file_png = (
    plot_dir /
    f"phase_set_site_concordance_after_best_flip_reverse_ecdf_by_sample_highlight90_100_{FIG_W_MM:.0f}x{FIG_H_MM:.0f}mm_insideLegend_min{min_shared_snps}snps_threshold50_PARincluded.png"
)
fig_file_pdf = (
    plot_dir /
    f"phase_set_site_concordance_after_best_flip_reverse_ecdf_by_sample_highlight90_100_{FIG_W_MM:.0f}x{FIG_H_MM:.0f}mm_insideLegend_min{min_shared_snps}snps_threshold50_PARincluded.pdf"
)
fig_file_svg = (
    plot_dir /
    f"phase_set_site_concordance_after_best_flip_reverse_ecdf_by_sample_highlight90_100_{FIG_W_MM:.0f}x{FIG_H_MM:.0f}mm_insideLegend_min{min_shared_snps}snps_threshold50_PARincluded.svg"
)

fig.savefig(fig_file_png, dpi=600, transparent=True)
fig.savefig(fig_file_pdf, transparent=True)
fig.savefig(fig_file_svg, transparent=True)

plt.show()

print(f"Saved plot: {fig_file_png}")
print(f"Saved plot: {fig_file_pdf}")
print(f"Saved plot: {fig_file_svg}")

# ------------------------------------------------------------
# Plot 2: mean across donors + shaded SD
# ------------------------------------------------------------

mean_curve = curve_df.mean(axis=1)
sd_curve = curve_df.std(axis=1)

overall_mean_conc = ecdf_summary["mean_concordance_percent"].mean()

highlight_colour = "#D62728"

mean_point_vals = {
    t: mean_curve.loc[t] if t in mean_curve.index else np.mean([
        np.mean(
            ecdf_df.loc[ecdf_df["sample"] == sample, metric_col].dropna().values >= t
        ) * 100
        for sample in sample_order
    ])
    for t in highlight_thresholds
}

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))

# This is the key fix: larger bottom margin for the two-line x-axis label
fig.subplots_adjust(
    left=0.22,
    right=0.98,
    bottom=0.36,
    top=0.88,
)

for t in highlight_thresholds:
    ax.axvline(
        t,
        color=highlight_colour,
        linestyle="--",
        linewidth=0.75,
        alpha=0.45,
        zorder=0,
    )

ax.plot(
    thresholds,
    mean_curve.values,
    color="#4C78A8",
    linewidth=1.45,
    label="Mean across donors",
)

ax.fill_between(
    thresholds,
    np.clip(mean_curve.values - sd_curve.values, 0, 100),
    np.clip(mean_curve.values + sd_curve.values, 0, 100),
    color="#4C78A8",
    alpha=0.20,
    linewidth=0,
    label="± SD",
)

ax.scatter(
    highlight_thresholds,
    [mean_point_vals[t] for t in highlight_thresholds],
    color=highlight_colour,
    s=26,
    zorder=5,
    edgecolor="white",
    linewidth=0.55,
)

label_box = dict(
    facecolor="none",
    edgecolor="none",
    alpha=0.8,
    pad=0.30,
)

# 90% label
ax.annotate(
    f"≥90% concordance\n{mean_point_vals[90]:.1f}% phase sets",
    xy=(90, mean_point_vals[90]),
    xytext=(84.5, 82.5),
    textcoords="data",
    arrowprops=dict(
        arrowstyle="-",
        lw=0.7,
        color=highlight_colour,
        shrinkA=0,
        shrinkB=4,
    ),
    fontsize=4.8,
    color=highlight_colour,
    ha="center",
    va="top",
    bbox=label_box,
    annotation_clip=False,
    clip_on=False,
)

# 100% label
ax.annotate(
    f"100% concordance\n{mean_point_vals[100]:.1f}% phase sets",
    xy=(100, mean_point_vals[100]),
    xytext=(102.0, 120.0),
    textcoords="data",
    arrowprops=dict(
        arrowstyle="-",
        lw=0.7,
        color=highlight_colour,
        shrinkA=0,
        shrinkB=4,
    ),
    fontsize=4.8,
    color=highlight_colour,
    ha="center",
    va="top",
    bbox=label_box,
    annotation_clip=False,
    clip_on=False,
)

ax.set_xlim(50, 112)
ax.set_ylim(0, 110)

ax.set_xlabel(
    "Within phase set concordance between\nWhatsHap and scDaisychain (%)",
    labelpad=2,
)
ax.set_ylabel(
    "Percent of phase sets ≥\nconcordance threshold",
    labelpad=2,
)

ax.set_xticks([50, 60, 70, 80, 90, 100])
ax.set_yticks([0, 25, 50, 75, 100])

ax.grid(axis="y", alpha=0.22, linewidth=0.4)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    loc="lower left",
    bbox_to_anchor=(0.03, 0.03),
    ncol=1,
    frameon=True,
    facecolor="white",
    edgecolor="0.8",
    framealpha=0.88,
    handlelength=1.3,
    handletextpad=0.4,
    borderpad=0.35,
    borderaxespad=0.0,
    labelspacing=0.35,
    prop={"size": 4.8},
)

mean_fig_file_png = (
    plot_dir /
    f"phase_set_site_concordance_after_best_flip_reverse_ecdf_mean_sd_highlight90_100_{FIG_W_MM:.0f}x{FIG_H_MM:.0f}mm_min{min_shared_snps}snps_threshold50_PARincluded.png"
)
mean_fig_file_pdf = (
    plot_dir /
    f"phase_set_site_concordance_after_best_flip_reverse_ecdf_mean_sd_highlight90_100_{FIG_W_MM:.0f}x{FIG_H_MM:.0f}mm_min{min_shared_snps}snps_threshold50_PARincluded.pdf"
)
mean_fig_file_svg = (
    plot_dir /
    f"phase_set_site_concordance_after_best_flip_reverse_ecdf_mean_sd_highlight90_100_{FIG_W_MM:.0f}x{FIG_H_MM:.0f}mm_min{min_shared_snps}snps_threshold50_PARincluded.svg"
)

fig.savefig(mean_fig_file_png, dpi=600, transparent=True)
fig.savefig(mean_fig_file_pdf, transparent=True)
fig.savefig(mean_fig_file_svg, transparent=True)

plt.show()

print(f"Saved plot: {mean_fig_file_png}")
print(f"Saved plot: {mean_fig_file_pdf}")
print(f"Saved plot: {mean_fig_file_svg}")

In [ ]:
# ------------------------------------------------------------
# Plot: site concordance after best flip by sample
# min 2 shared SNPs
# separate weighted and unweighted plots
# 90 x 60 mm, Illustrator-editable text
# ------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Illustrator-editable / publication settings
# ------------------------------------------------------------

mpl.rcParams.update({
    # Keep text editable in Illustrator
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    # Common editable font
    "font.family": "Arial",

    # Legible for 90 x 60 mm
    "font.size": 6.2,
    "axes.labelsize": 7.0,
    "xtick.labelsize": 5.8,
    "ytick.labelsize": 6.0,
    "axes.titlesize": 7.0,

    # Fine line weights
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.4,
    "ytick.major.size": 2.4,

    # Transparent output
    "savefig.transparent": True,
})

MM_TO_INCH = 1 / 25.4
FIG_W = 90 * MM_TO_INCH
FIG_H = 50 * MM_TO_INCH

metric_col = "site_concordance_after_best_flip"
weight_col = "n_shared_snps"
min_shared_snps = 2

sample_labels = {
    "DRG_0198_01": "DRG_0198_01",
    "DRG_0199_01": "DRG_0199_01",
    "DRG_0341_01": "DRG_0341_01",
    "DRG_1409_01": "DRG_1409_01",
    "DRG_1215_02": "DRG_1215_02",
    "DRG_2324_02": "DRG_2324_02",
    "DRG_2902_02": "DRG_2902_02",
    "DRG_5467_02": "DRG_5467_02",
}

# ------------------------------------------------------------
# Filter to usable phase sets
# ------------------------------------------------------------

site_df = plot_df.dropna(
    subset=["sample", metric_col, weight_col]
).copy()

site_df = site_df[site_df[weight_col] >= min_shared_snps].copy()

if site_df.empty:
    raise ValueError(
        f"No phase sets left after filtering to {weight_col} >= {min_shared_snps}"
    )

# ------------------------------------------------------------
# Summarise by sample
# ------------------------------------------------------------

summary_rows = []

for sample in samples:
    sub = site_df[site_df["sample"] == sample].copy()

    if sub.empty:
        continue

    weighted_site_concordance = np.average(
        sub[metric_col],
        weights=sub[weight_col],
    )

    unweighted_site_concordance = sub[metric_col].mean()

    row = {
        "sample": sample,
        "sample_label": sample_labels.get(sample, sample),
        "n_phase_sets": len(sub),
        "total_shared_snps": sub[weight_col].sum(),
        "weighted_site_concordance": weighted_site_concordance,
        "unweighted_site_concordance": unweighted_site_concordance,
    }

    # Optional count-based check
    if {
        "n_concordant_sites_after_best_flip",
        "n_shared_snps",
    }.issubset(sub.columns):
        row["count_based_site_concordance"] = (
            sub["n_concordant_sites_after_best_flip"].sum()
            / sub["n_shared_snps"].sum()
        )

    summary_rows.append(row)

summary_site_min2 = pd.DataFrame(summary_rows)

summary_site_min2 = summary_site_min2.sort_values(
    "weighted_site_concordance",
    ascending=False,
).reset_index(drop=True)

summary_file = (
    plot_dir /
    "site_concordance_after_best_flip_by_sample_min2snps_weighted_unweighted.tsv"
)
summary_site_min2.to_csv(summary_file, sep="\t", index=False)

print(summary_site_min2)
print(f"Saved summary: {summary_file}")

# ------------------------------------------------------------
# Helper for plotting
# ------------------------------------------------------------

def plot_site_concordance_bar(
    df,
    value_col,
    ylabel,
    outfile_prefix,
    sort_col=None,
):
    """
    Make exact 90 x 60 mm bar plot with editable text.

    Adds dashed red line showing mean across samples.
    Do not use bbox_inches='tight', because it changes the exported PDF/SVG
    page size and can cause inconsistent sizing in Illustrator.
    """

    if sort_col is None:
        sort_col = value_col

    plot_data = (
        df
        .sort_values(sort_col, ascending=False)
        .reset_index(drop=True)
        .copy()
    )

    x = np.arange(len(plot_data))
    vals = plot_data[value_col] * 100

    # Mean across samples, using the plotted per-sample values
    mean_across_samples = vals.mean()

    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))

    # Leave room for rotated x labels and value labels
    fig.subplots_adjust(
        left=0.16,
        right=0.98,
        bottom=0.28,
        top=0.92,
    )

    ax.bar(
        x,
        vals,
        color="#4C78A8",
        width=0.72,
        linewidth=0,
        alpha=0.95,
        zorder=3,
    )

    # ---------- dashed red mean line ----------
    # ---------- dashed red mean line ----------
    ax.axhline(
        mean_across_samples,
        color="#D62728",
        linestyle="--",
        linewidth=0.9,
        zorder=5,
        label=f"Mean = {mean_across_samples:.1f}%",
    )

    # Legend inside bottom-left of plot area
    ax.legend(
        loc="lower left",
        bbox_to_anchor=(0.03, 0.04),
        frameon=True,
        facecolor="white",
        edgecolor="0.8",
        framealpha=0.85,
        handlelength=1.8,
        handletextpad=0.45,
        borderpad=0.35,
        borderaxespad=0.0,
        fontsize=5.6,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        plot_data["sample_label"],
        rotation=45,
        ha="right",
        rotation_mode="anchor",
    )

    ax.set_ylabel(ylabel, labelpad=2)
    ax.set_xlabel("Sample", labelpad=2)

    ax.set_ylim(0, 105)
    ax.set_yticks([0, 25, 50, 75, 100])

    ax.grid(axis="y", alpha=0.22, linewidth=0.4)
    ax.set_axisbelow(True)

    # Cleaner spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        top=False,
        right=False,
        direction="out",
        pad=1.5,
    )

    # Value labels
    for i, val in enumerate(vals):
        if pd.notna(val):
            ax.text(
                i,
                min(val + 1.2, 103.5),
                f"{val:.1f}",
                ha="center",
                va="bottom",
                fontsize=5.5,
                rotation=0,
                clip_on=False,
                zorder=6,
            )

    fig_file_png = plot_dir / f"{outfile_prefix}_90x60mm.png"
    fig_file_pdf = plot_dir / f"{outfile_prefix}_90x60mm.pdf"
    fig_file_svg = plot_dir / f"{outfile_prefix}_90x60mm.svg"

    fig.savefig(fig_file_png, dpi=600)
    fig.savefig(fig_file_pdf)
    fig.savefig(fig_file_svg)

    plt.show()

    print(f"Saved plot: {fig_file_png}")
    print(f"Saved plot: {fig_file_pdf}")
    print(f"Saved plot: {fig_file_svg}")


# ------------------------------------------------------------
# Plot 1: weighted site concordance
# ------------------------------------------------------------

plot_site_concordance_bar(
    df=summary_site_min2,
    value_col="weighted_site_concordance",
    ylabel="Weighted site concordance (%)",
    outfile_prefix=(
        f"weighted_site_concordance_after_best_flip_by_sample_"
        f"min{min_shared_snps}snps"
    ),
)

# ------------------------------------------------------------
# Plot 2: unweighted site concordance
# ------------------------------------------------------------

plot_site_concordance_bar(
    df=summary_site_min2,
    value_col="unweighted_site_concordance",
    ylabel="Average within\nphase set concordance (%)",
    outfile_prefix=(
        f"unweighted_site_concordance_after_best_flip_by_sample_"
        f"min{min_shared_snps}snps"
    ),
)

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Illustrator-editable / publication settings
# ------------------------------------------------------------

mpl.rcParams.update({
    # Keep text editable in Illustrator
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    # Common editable font
    "font.family": "Arial",

    # Small but legible for 90 x 60 mm faceted plots
    "font.size": 5.6,
    "axes.labelsize": 6.0,
    "axes.titlesize": 5.6,
    "xtick.labelsize": 5.0,
    "ytick.labelsize": 5.0,
    "legend.fontsize": 5.0,

    # Fine line weights
    "axes.linewidth": 0.55,
    "xtick.major.width": 0.55,
    "ytick.major.width": 0.55,
    "xtick.major.size": 2.0,
    "ytick.major.size": 2.0,

    # Transparent output for Illustrator
    "savefig.transparent": True,
})

MM_TO_INCH = 1 / 25.4
FIG_W = 90 * MM_TO_INCH
FIG_H = 60 * MM_TO_INCH

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

samples = [
    # add your samples here
    "DRG_0198_01",
    "DRG_0199_01",
    "DRG_0341_01",
    "DRG_1409_01",
    "DRG_1215_02",
    "DRG_2324_02",
    "DRG_2902_02",
    "DRG_5467_02",
]

sample_labels = {
    "DRG_0198_01": "DRG_0198_01",
    "DRG_0199_01": "DRG_0199_01",
    "DRG_0341_01": "DRG_0341_01",
    "DRG_1409_01": "DRG_1409_01",
    "DRG_1215_02": "DRG_1215_02",
    "DRG_2324_02": "DRG_2324_02",
    "DRG_2902_02": "DRG_2902_02",
    "DRG_5467_02": "DRG_5467_02",
}

root = Path(
    "/home/913/dk4874/scratch/gdata/scDaisychain_paper/human/output/"
    "scDaisychain_modes_multi_filtered_weighted"
)

orig_matrix_dirname = "daisychain_split_bams_count"
corr_matrix_dirname = "whatshap_corrected_qual20gq10_daisychain_split_bams_count"

outdir = root / "whatshap_corrected_qual20gq10_daisychain_comparison_all_samples"
outdir.mkdir(parents=True, exist_ok=True)

min_classified_per_cell = 10
min_classified_per_gene = 10

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def read_matrix(path):
    print(f"Reading {path}")
    df = pd.read_csv(path, index_col=0)
    df = df.apply(pd.to_numeric, errors="coerce").fillna(0)
    return df


def xi_fraction(xi, xa):
    total = xi + xa
    return xi / total.replace(0, np.nan)


def savefig(fig, name):
    """
    Save with exact 90 x 60 mm artboard/page size.

    Do not use bbox_inches='tight', because that changes the exported
    PDF/SVG page size and can make Illustrator sizing inconsistent.
    """
    for ext in ["png", "pdf", "svg"]:
        if ext == "png":
            fig.savefig(outdir / f"{name}.{ext}", dpi=600)
        else:
            fig.savefig(outdir / f"{name}.{ext}")


def make_facet_grid(n, ncols=4, sharex=True, sharey=True):
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(FIG_W, FIG_H),
        squeeze=False,
        sharex=sharex,
        sharey=sharey,
    )

    axes = axes.ravel()

    # Roomier layout to avoid cutoff at the top, right, and bottom.
    fig.subplots_adjust(
        left=0.135,
        right=0.965,
        bottom=0.185,
        top=0.835,
        wspace=0.22,
        hspace=0.62,
    )

    return fig, axes


def corr_stats(df, x_col="orig_Xi_fraction", y_col="corr_Xi_fraction"):
    """
    Return Pearson and Spearman correlations between original and corrected Xi fraction.
    Returns NaN if fewer than 2 valid paired observations exist.
    """
    sub = df[[x_col, y_col]].dropna().copy()

    if len(sub) < 2:
        return np.nan, np.nan

    pearson_r = sub[x_col].corr(sub[y_col], method="pearson")
    spearman_rho = sub[x_col].corr(sub[y_col], method="spearman")

    return pearson_r, spearman_rho


def fmt_corr(x):
    if pd.isna(x):
        return "NA"
    return f"{x:.2f}"


def clean_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        top=False,
        right=False,
        labeltop=False,
        labelright=False,
        direction="out",
        pad=1.2,
    )

    ax.grid(axis="both", alpha=0.16, linewidth=0.35)


def format_facets(fig, axes, n_used, ncols, xlabel, ylabel):
    """
    Hide internal tick labels and add one shared x/y label for the whole figure.
    """
    nrows = math.ceil(n_used / ncols)

    for i, ax in enumerate(axes[:n_used]):
        row = i // ncols
        col = i % ncols

        # Remove repeated per-panel axis labels
        ax.set_xlabel("")
        ax.set_ylabel("")

        # Only bottom row gets x tick labels
        if row != nrows - 1:
            ax.tick_params(labelbottom=False)

        # Only first column gets y tick labels
        if col != 0:
            ax.tick_params(labelleft=False)

    # Shared figure-level labels, moved inward to avoid clipping
    fig.text(
        0.56,
        0.060,
        xlabel,
        ha="center",
        va="center",
        fontsize=6.4,
    )

    fig.text(
        0.040,
        0.51,
        ylabel,
        ha="center",
        va="center",
        rotation="vertical",
        fontsize=6.4,
    )


# ------------------------------------------------------------
# Per-sample processing
# ------------------------------------------------------------

all_cell = []
all_gene = []
sample_summary_rows = []

for sample in samples:
    print(f"\n==================== {sample} ====================")

    base = root / sample

    orig_dir = base / orig_matrix_dirname / "matrices"
    corr_dir = base / corr_matrix_dirname / "matrices"

    orig_xa_file = orig_dir / "Xa.csv"
    orig_xi_file = orig_dir / "Xi.csv"
    corr_xa_file = corr_dir / "Xa.csv"
    corr_xi_file = corr_dir / "Xi.csv"

    required_files = [orig_xa_file, orig_xi_file, corr_xa_file, corr_xi_file]
    missing = [str(f) for f in required_files if not f.exists()]

    if missing:
        print(f"[WARN] Skipping {sample}; missing files:")
        for f in missing:
            print(f"  {f}")
        continue

    # Read
    orig_xa = read_matrix(orig_xa_file)
    orig_xi = read_matrix(orig_xi_file)
    corr_xa = read_matrix(corr_xa_file)
    corr_xi = read_matrix(corr_xi_file)

    # Align genes/cells within this sample
    common_genes = (
        orig_xa.index
        .intersection(orig_xi.index)
        .intersection(corr_xa.index)
        .intersection(corr_xi.index)
    )

    common_cells = (
        orig_xa.columns
        .intersection(orig_xi.columns)
        .intersection(corr_xa.columns)
        .intersection(corr_xi.columns)
    )

    print(f"Common genes: {len(common_genes):,}")
    print(f"Common cells: {len(common_cells):,}")

    orig_xa = orig_xa.loc[common_genes, common_cells]
    orig_xi = orig_xi.loc[common_genes, common_cells]
    corr_xa = corr_xa.loc[common_genes, common_cells]
    corr_xi = corr_xi.loc[common_genes, common_cells]

    # --------------------------------------------------------
    # Per-cell Xi fraction
    # --------------------------------------------------------

    cell_orig_xa = orig_xa.sum(axis=0)
    cell_orig_xi = orig_xi.sum(axis=0)
    cell_corr_xa = corr_xa.sum(axis=0)
    cell_corr_xi = corr_xi.sum(axis=0)

    cell_df = pd.DataFrame({
        "sample": sample,
        "cell": common_cells,
        "orig_Xa": cell_orig_xa.values,
        "orig_Xi": cell_orig_xi.values,
        "corr_Xa": cell_corr_xa.values,
        "corr_Xi": cell_corr_xi.values,
    })

    cell_df["orig_classified"] = cell_df["orig_Xa"] + cell_df["orig_Xi"]
    cell_df["corr_classified"] = cell_df["corr_Xa"] + cell_df["corr_Xi"]

    cell_df["orig_Xi_fraction"] = (
        cell_df["orig_Xi"] / cell_df["orig_classified"].replace(0, np.nan)
    )
    cell_df["corr_Xi_fraction"] = (
        cell_df["corr_Xi"] / cell_df["corr_classified"].replace(0, np.nan)
    )
    cell_df["delta_Xi_fraction"] = (
        cell_df["corr_Xi_fraction"] - cell_df["orig_Xi_fraction"]
    )

    cell_df_filt = cell_df[
        (cell_df["orig_classified"] >= min_classified_per_cell)
        & (cell_df["corr_classified"] >= min_classified_per_cell)
        & cell_df["orig_Xi_fraction"].notna()
        & cell_df["corr_Xi_fraction"].notna()
    ].copy()

    all_cell.append(cell_df_filt)

    # --------------------------------------------------------
    # Per-gene Xi fraction
    # --------------------------------------------------------

    gene_orig_xa = orig_xa.sum(axis=1)
    gene_orig_xi = orig_xi.sum(axis=1)
    gene_corr_xa = corr_xa.sum(axis=1)
    gene_corr_xi = corr_xi.sum(axis=1)

    gene_df = pd.DataFrame({
        "sample": sample,
        "gene": common_genes,
        "orig_Xa": gene_orig_xa.values,
        "orig_Xi": gene_orig_xi.values,
        "corr_Xa": gene_corr_xa.values,
        "corr_Xi": gene_corr_xi.values,
    })

    gene_df["orig_classified"] = gene_df["orig_Xa"] + gene_df["orig_Xi"]
    gene_df["corr_classified"] = gene_df["corr_Xa"] + gene_df["corr_Xi"]

    gene_df["orig_Xi_fraction"] = (
        gene_df["orig_Xi"] / gene_df["orig_classified"].replace(0, np.nan)
    )
    gene_df["corr_Xi_fraction"] = (
        gene_df["corr_Xi"] / gene_df["corr_classified"].replace(0, np.nan)
    )
    gene_df["delta_Xi_fraction"] = (
        gene_df["corr_Xi_fraction"] - gene_df["orig_Xi_fraction"]
    )
    gene_df["abs_delta_Xi_fraction"] = gene_df["delta_Xi_fraction"].abs()

    gene_df_filt = gene_df[
        (gene_df["orig_classified"] >= min_classified_per_gene)
        & (gene_df["corr_classified"] >= min_classified_per_gene)
        & gene_df["orig_Xi_fraction"].notna()
        & gene_df["corr_Xi_fraction"].notna()
    ].copy()

    all_gene.append(gene_df_filt)

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    cell_pearson_r, cell_spearman_rho = corr_stats(cell_df_filt)
    gene_pearson_r, gene_spearman_rho = corr_stats(gene_df_filt)

    sample_summary_rows.append({
        "sample": sample,
        "common_genes": len(common_genes),
        "common_cells": len(common_cells),
        "cells_min10": len(cell_df_filt),
        "genes_min10": len(gene_df_filt),
        "median_cell_delta_Xi_fraction": cell_df_filt["delta_Xi_fraction"].median(),
        "mean_cell_delta_Xi_fraction": cell_df_filt["delta_Xi_fraction"].mean(),
        "median_gene_delta_Xi_fraction": gene_df_filt["delta_Xi_fraction"].median(),
        "mean_gene_delta_Xi_fraction": gene_df_filt["delta_Xi_fraction"].mean(),
        "cell_pearson_r_orig_vs_corr": cell_pearson_r,
        "cell_spearman_rho_orig_vs_corr": cell_spearman_rho,
        "gene_pearson_r_orig_vs_corr": gene_pearson_r,
        "gene_spearman_rho_orig_vs_corr": gene_spearman_rho,
    })

# ------------------------------------------------------------
# Combine and save
# ------------------------------------------------------------

if not all_cell:
    raise ValueError("No per-cell data loaded. Check paths/files.")

if not all_gene:
    raise ValueError("No per-gene data loaded. Check paths/files.")

cell_all = pd.concat(all_cell, ignore_index=True)
gene_all = pd.concat(all_gene, ignore_index=True)
summary = pd.DataFrame(sample_summary_rows)

cell_all.to_csv(
    outdir / "all_samples_per_cell_xi_fraction_min10_comparison.tsv",
    sep="\t",
    index=False,
)

gene_all.to_csv(
    outdir / "all_samples_per_gene_xi_fraction_min10_comparison.tsv",
    sep="\t",
    index=False,
)

summary.to_csv(
    outdir / "all_samples_xi_fraction_comparison_summary.tsv",
    sep="\t",
    index=False,
)

print("\nSummary:")
print(summary.to_string(index=False))

# ------------------------------------------------------------
# Correlation summary table
# ------------------------------------------------------------

corr_rows = []

for sample in samples:
    sub_cell = cell_all[cell_all["sample"] == sample].copy()
    sub_gene = gene_all[gene_all["sample"] == sample].copy()

    if len(sub_cell) > 0:
        pearson_r, spearman_rho = corr_stats(sub_cell)
        corr_rows.append({
            "sample": sample,
            "level": "cell",
            "n": len(sub_cell),
            "pearson_r": pearson_r,
            "spearman_rho": spearman_rho,
        })

    if len(sub_gene) > 0:
        pearson_r, spearman_rho = corr_stats(sub_gene)
        corr_rows.append({
            "sample": sample,
            "level": "gene",
            "n": len(sub_gene),
            "pearson_r": pearson_r,
            "spearman_rho": spearman_rho,
        })

corr_summary = pd.DataFrame(corr_rows)

corr_summary.to_csv(
    outdir / "all_samples_orig_vs_corrected_xi_fraction_correlations.tsv",
    sep="\t",
    index=False,
)

print("\nOriginal vs corrected Xi-fraction correlations:")
print(corr_summary.to_string(index=False))

# ------------------------------------------------------------
# Top corrected Xi-fraction genes per sample
# ------------------------------------------------------------

top_genes = (
    gene_all
    .sort_values(
        ["sample", "corr_Xi_fraction", "corr_classified"],
        ascending=[True, False, False],
    )
    .groupby("sample", group_keys=False)
    .head(50)
    [
        [
            "sample",
            "gene",
            "corr_Xi_fraction",
            "orig_Xi_fraction",
            "delta_Xi_fraction",
            "corr_Xi",
            "corr_Xa",
            "corr_classified",
            "orig_classified",
        ]
    ]
)

top_genes.to_csv(
    outdir / "top_corrected_Xi_fraction_genes_per_sample_min10.tsv",
    sep="\t",
    index=False,
)

print("\nTop corrected Xi-fraction genes per sample:")
print(top_genes.groupby("sample").head(10).to_string(index=False))

# ------------------------------------------------------------
# Faceted plot 1: Xi fraction per cell, original vs corrected
# with correlation
# ------------------------------------------------------------

plot_samples = [s for s in samples if s in set(cell_all["sample"])]
ncols = 4

fig, axes = make_facet_grid(
    len(plot_samples),
    ncols=ncols,
    sharex=True,
    sharey=True,
)

for ax, sample in zip(axes, plot_samples):
    sub = cell_all[cell_all["sample"] == sample].copy()
    pearson_r, spearman_rho = corr_stats(sub)

    ax.scatter(
        sub["orig_Xi_fraction"],
        sub["corr_Xi_fraction"],
        alpha=0.35,
        s=2.8,
        linewidths=0,
        color="#4C78A8",
    )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=0.6,
        color="black",
        alpha=0.65,
    )

    ax.text(
        0.04,
        0.96,
        f"r={fmt_corr(pearson_r)}\nρ={fmt_corr(spearman_rho)}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=4.2,
        bbox=dict(
            boxstyle="round,pad=0.12",
            facecolor="white",
            edgecolor="none",
            alpha=0.75,
        ),
    )

    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_xticks([0, 0.5, 1.0])
    ax.set_yticks([0, 0.5, 1.0])

    ax.set_title(
        f"{sample_labels.get(sample, sample)}\nN={len(sub):,}",
        pad=1.5,
        fontsize=5.2,
    )

    clean_axis(ax)

for ax in axes[len(plot_samples):]:
    ax.axis("off")

format_facets(
    fig,
    axes,
    n_used=len(plot_samples),
    ncols=ncols,
    xlabel="Original Xi fraction",
    ylabel="Corrected Xi fraction",
)

savefig(
    fig,
    "faceted_xi_fraction_per_cell_original_vs_corrected_min10_with_correlation_90x60mm_shared_axes_nocutoff",
)
plt.show()

# ------------------------------------------------------------
# Faceted plot 2: delta Xi fraction per cell
# ------------------------------------------------------------

fig, axes = make_facet_grid(
    len(plot_samples),
    ncols=ncols,
    sharex=True,
    sharey=True,
)

bins = np.linspace(-1, 1, 81)

for ax, sample in zip(axes, plot_samples):
    sub = cell_all[cell_all["sample"] == sample].copy()

    ax.hist(
        sub["delta_Xi_fraction"].dropna(),
        bins=bins,
        edgecolor="black",
        linewidth=0.15,
        color="#4C78A8",
        alpha=0.85,
    )

    ax.axvline(
        0,
        linestyle="--",
        linewidth=0.6,
        color="black",
        alpha=0.75,
    )

    ax.set_xlim(-1, 1)
    ax.set_xticks([-1, 0, 1])

    ax.set_title(
        f"{sample_labels.get(sample, sample)}\nN={len(sub):,}",
        pad=1.5,
        fontsize=5.2,
    )

    clean_axis(ax)

for ax in axes[len(plot_samples):]:
    ax.axis("off")

format_facets(
    fig,
    axes,
    n_used=len(plot_samples),
    ncols=ncols,
    xlabel="Corrected − original Xi fraction",
    ylabel="Cells",
)

savefig(
    fig,
    "faceted_delta_xi_fraction_per_cell_min10_90x60mm_shared_axes_nocutoff",
)
plt.show()

# ------------------------------------------------------------
# Faceted plot 3: Xi fraction per gene, original vs corrected
# with correlation
# ------------------------------------------------------------

plot_samples = [s for s in samples if s in set(gene_all["sample"])]

fig, axes = make_facet_grid(
    len(plot_samples),
    ncols=ncols,
    sharex=True,
    sharey=True,
)

for ax, sample in zip(axes, plot_samples):
    sub = gene_all[gene_all["sample"] == sample].copy()
    pearson_r, spearman_rho = corr_stats(sub)

    ax.scatter(
        sub["orig_Xi_fraction"],
        sub["corr_Xi_fraction"],
        alpha=0.55,
        s=5.5,
        linewidths=0,
        color="#4C78A8",
    )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=0.6,
        color="black",
        alpha=0.65,
    )

    ax.text(
        0.96,
        0.04,
        f"r={fmt_corr(pearson_r)}\nρ={fmt_corr(spearman_rho)}",
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=4.2,
        bbox=dict(
            boxstyle="round,pad=0.12",
            facecolor="white",
            edgecolor="none",
            alpha=0.75,
        ),
    )

    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_xticks([0, 0.5, 1.0])
    ax.set_yticks([0, 0.5, 1.0])

    ax.set_title(
        f"{sample_labels.get(sample, sample)}\nN={len(sub):,}",
        pad=1.5,
        fontsize=5.2,
    )

    clean_axis(ax)

for ax in axes[len(plot_samples):]:
    ax.axis("off")

format_facets(
    fig,
    axes,
    n_used=len(plot_samples),
    ncols=ncols,
    xlabel="Original Xi fraction",
    ylabel="Corrected Xi fraction",
)

savefig(
    fig,
    "faceted_xi_fraction_per_gene_original_vs_corrected_min10_with_correlation_90x60mm_shared_axes_nocutoff",
)
plt.show()

# ------------------------------------------------------------
# Faceted plot 4: delta Xi fraction per gene
# ------------------------------------------------------------

fig, axes = make_facet_grid(
    len(plot_samples),
    ncols=ncols,
    sharex=True,
    sharey=True,
)

bins = np.linspace(-1, 1, 81)

for ax, sample in zip(axes, plot_samples):
    sub = gene_all[gene_all["sample"] == sample].copy()

    ax.hist(
        sub["delta_Xi_fraction"].dropna(),
        bins=bins,
        edgecolor="black",
        linewidth=0.15,
        color="#4C78A8",
        alpha=0.85,
    )

    ax.axvline(
        0,
        linestyle="--",
        linewidth=0.6,
        color="black",
        alpha=0.75,
    )

    ax.set_xlim(-1, 1)
    ax.set_xticks([-1, 0, 1])

    ax.set_title(
        f"{sample_labels.get(sample, sample)}\nN={len(sub):,}",
        pad=1.5,
        fontsize=5.2,
    )

    clean_axis(ax)

for ax in axes[len(plot_samples):]:
    ax.axis("off")

format_facets(
    fig,
    axes,
    n_used=len(plot_samples),
    ncols=ncols,
    xlabel="Corrected − original Xi fraction",
    ylabel="Genes",
)

savefig(
    fig,
    "faceted_delta_xi_fraction_per_gene_min10_90x60mm_shared_axes_nocutoff",
)
plt.show()

print(f"\nSaved all outputs to:\n{outdir}")